# Micro-Expression Spotting — Clip Output

Notebook utama untuk spotting.

Output:
- `/output/apex/metadata.xlsx`
- `/output/apex/dataset/...`

Struktur folder input dipertahankan pada `/output/apex/dataset`.


In [ ]:
import gc
import re
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path

import cv2
import dlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from apex import ApexPhase, ApexSmoother
from features_extraction.poc import POC
from features_extraction.quadran import Quadran
from features_extraction.vektor import Vektor
from spotting import SpottingConfig

In [ ]:
config = SpottingConfig(
    dataset_root=PROJECT_ROOT / 'dataset',
    output_root=PROJECT_ROOT / 'output' / 'apex',
    predictor_path=Path('./models/shape_predictor_68_face_landmarks.dat'),
    video_extensions={'.avi', '.mp4', '.mov', '.mkv', '.mpeg'},
    overwrite=False,
)

print(f'DATASET_ROOT = {config.dataset_root}')
print(f'OUTPUT_DATASET_ROOT = {config.output_dataset_root}')
print(f'METADATA_PATH = {config.metadata_path}')

In [ ]:
@dataclass(slots=True)
class VideoSample:
    video_path: Path
    relative_parent: Path
    stem: str

    @property
    def relative_video_key(self) -> Path:
        return self.relative_parent / self.stem


def natural_sort_key(value: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', value)]


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def discover_video_samples(dataset_root: Path, video_extensions: set[str]) -> list[VideoSample]:
    samples = []
    for video_path in sorted(dataset_root.rglob('*'), key=lambda p: natural_sort_key(str(p))):
        if not video_path.is_file():
            continue
        if video_path.suffix.lower() not in video_extensions:
            continue
        rel = video_path.relative_to(dataset_root)
        samples.append(VideoSample(video_path=video_path, relative_parent=rel.parent, stem=video_path.stem))
    return samples


def extract_region(image: np.ndarray, landmarks, indices: list[int]) -> np.ndarray:
    pts = [(landmarks.part(i).x, landmarks.part(i).y) for i in indices]
    xs, ys = zip(*pts)
    left = max(0, min(xs) - config.padding_x)
    top = max(0, min(ys) - config.padding_y)
    right = min(image.shape[1], max(xs) + config.padding_x)
    bottom = min(image.shape[0], max(ys) + config.padding_y)
    return image[top:bottom, left:right]


def detect_landmarks(gray: np.ndarray, detector, predictor):
    faces = detector(gray)
    if len(faces) > 0:
        return predictor(gray, faces[0])
    return None


def compute_frame_magnitude(prev_gray, prev_lm, curr_gray, curr_lm) -> float:
    roi_mags = []
    for region_name, lm_indices in config.regions.items():
        try:
            roi_prev = extract_region(prev_gray, prev_lm, lm_indices)
            roi_curr = extract_region(curr_gray, curr_lm, lm_indices)
            if roi_prev.size == 0 or roi_curr.size == 0:
                return 0.0
            target = config.target_size[region_name]
            roi_prev = cv2.resize(roi_prev, target)
            roi_curr = cv2.resize(roi_curr, target)
            poc = POC(roi_prev, roi_curr, config.block_size)
            vec = Vektor(poc.getPOC(), config.block_size)
            quad_data = Quadran(vec.getVektor()).getQuadran()
            magnitudes = [float(block[4]) for block in quad_data]
            roi_mags.append(np.mean(magnitudes) if magnitudes else 0.0)
        except Exception:
            return 0.0
    return float(np.mean(roi_mags)) if roi_mags else 0.0


def load_video_frames(video_path: Path) -> list[np.ndarray]:
    frames = []
    cap = cv2.VideoCapture(str(video_path))
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames


def build_magnitude_signal(frames: list[np.ndarray], detector, predictor) -> list[float]:
    if len(frames) < 3:
        return []
    grays = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frames]
    magnitudes = []
    prev_gray = grays[0]
    prev_lm = detect_landmarks(prev_gray, detector, predictor)
    for curr_gray in grays[1:]:
        curr_lm = detect_landmarks(curr_gray, detector, predictor)
        if prev_lm is not None and curr_lm is not None:
            mag = compute_frame_magnitude(prev_gray, prev_lm, curr_gray, curr_lm)
        else:
            mag = 0.0
        magnitudes.append(float(mag))
        prev_gray = curr_gray
        prev_lm = curr_lm
    return magnitudes


def detect_events(magnitudes: list[float]) -> tuple[list[float], list[dict], dict]:
    smoothed = ApexSmoother.smooth(magnitudes)
    signal_arr = np.asarray(smoothed, dtype=float)
    height_threshold = float(signal_arr.mean() + signal_arr.std())
    detector = ApexPhase(
        distance_threshold=config.distance_threshold,
        prominence_threshold=config.prominence_threshold,
        cutoff_ratio=config.cutoff_ratio,
    )
    apex_indices = detector.find_top_k_apex(smoothed, k=config.top_k_apex, height=height_threshold)
    phases = detector.find_phase(smoothed, apex_indices) if apex_indices else {}
    events = []
    for event_no, apex_idx in enumerate(apex_indices, start=1):
        phase = phases[apex_idx]
        duration = int(phase['end'] - phase['start'])
        if duration < config.min_phase_duration:
            continue
        events.append({
            'event_no': event_no,
            'onset_signal': int(phase['start']),
            'apex_signal': int(apex_idx),
            'offset_signal': int(phase['end']),
            'duration': duration,
        })
    window_length = ApexSmoother.calculate_window_length(len(magnitudes))
    meta = {
        'window_length': window_length,
        'polyorder': ApexSmoother.calculate_polyorder(window_length),
        'height_threshold': height_threshold,
    }
    return smoothed, events, meta


def signal_index_to_frame_index(signal_index: int) -> int:
    return int(signal_index + 1)


def save_event_clip_frames(frames: list[np.ndarray], sample_output_dir: Path, events: list[dict]) -> list[dict]:
    ensure_dir(sample_output_dir)
    rows = []
    for event in events:
        onset_frame = min(signal_index_to_frame_index(event['onset_signal']), len(frames) - 1)
        apex_frame = min(signal_index_to_frame_index(event['apex_signal']), len(frames) - 1)
        offset_frame = min(signal_index_to_frame_index(event['offset_signal']), len(frames) - 1)
        clip_dir = sample_output_dir / f"event_{event['event_no']:02d}"
        ensure_dir(clip_dir)
        write_idx = 0
        for frame_idx in range(onset_frame, offset_frame + 1):
            out_path = clip_dir / f'frame_{write_idx:05d}.jpg'
            cv2.imwrite(str(out_path), frames[frame_idx])
            write_idx += 1
        rows.append({
            'relative_sample': str(sample_output_dir.relative_to(config.output_dataset_root)),
            'event_no': event['event_no'],
            'onset_signal': event['onset_signal'],
            'apex_signal': event['apex_signal'],
            'offset_signal': event['offset_signal'],
            'onset_frame': onset_frame,
            'apex_frame': apex_frame,
            'offset_frame': offset_frame,
            'duration': event['duration'],
            'saved_frames': write_idx,
            'clip_dir': str(clip_dir.relative_to(config.output_root)),
        })
    return rows


In [ ]:
if not config.dataset_root.exists():
    raise FileNotFoundError(f'Dataset root not found: {config.dataset_root}')
if not config.predictor_path.exists():
    raise FileNotFoundError(f'Predictor not found: {config.predictor_path}')

ensure_dir(config.output_root)
ensure_dir(config.output_dataset_root)
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(str(config.predictor_path))
samples = discover_video_samples(config.dataset_root, config.video_extensions)
print(f'Found {len(samples)} video files')
samples[:5]

In [ ]:
metadata_rows = []
for idx, sample in enumerate(samples, start=1):
    relative_key = sample.relative_video_key
    sample_output_dir = config.output_dataset_root / relative_key

    if sample_output_dir.exists() and not config.overwrite:
        print(f'[{idx}/{len(samples)}] skip {relative_key}')
        continue

    if sample_output_dir.exists() and config.overwrite:
        shutil.rmtree(sample_output_dir)

    print(f'[{idx}/{len(samples)}] process {relative_key}')
    frames = load_video_frames(sample.video_path)
    if len(frames) < 3:
        print(f'  skip: too few frames ({len(frames)})')
        continue

    magnitudes = build_magnitude_signal(frames, detector, predictor)
    if len(magnitudes) < 5:
        print(f'  skip: too few magnitudes ({len(magnitudes)})')
        continue

    try:
        _smoothed, events, meta = detect_events(magnitudes)
    except ValueError as e:
        print(f'  skip: {e}')
        continue

    rows = save_event_clip_frames(frames, sample_output_dir, events)
    if not rows:
        metadata_rows.append({
            'relative_sample': str(relative_key),
            'source_video': str(sample.video_path),
            'event_no': None,
            'onset_signal': None,
            'apex_signal': None,
            'offset_signal': None,
            'onset_frame': None,
            'apex_frame': None,
            'offset_frame': None,
            'duration': None,
            'saved_frames': 0,
            'window_length': meta['window_length'],
            'polyorder': meta['polyorder'],
            'height_threshold': meta['height_threshold'],
            'clip_dir': None,
        })
    else:
        for row in rows:
            metadata_rows.append({
                'relative_sample': row['relative_sample'],
                'source_video': str(sample.video_path),
                'event_no': row['event_no'],
                'onset_signal': row['onset_signal'],
                'apex_signal': row['apex_signal'],
                'offset_signal': row['offset_signal'],
                'onset_frame': row['onset_frame'],
                'apex_frame': row['apex_frame'],
                'offset_frame': row['offset_frame'],
                'duration': row['duration'],
                'saved_frames': row['saved_frames'],
                'window_length': meta['window_length'],
                'polyorder': meta['polyorder'],
                'height_threshold': meta['height_threshold'],
                'clip_dir': row['clip_dir'],
            })
    gc.collect()

metadata_df = pd.DataFrame(metadata_rows)
metadata_df.to_excel(config.metadata_path, index=False)
print(f'Saved metadata: {config.metadata_path}')
print(metadata_df.head())
metadata_df